In [1]:
# data setup

import pandas as pd
import numpy as np

# TODO: Improve Automation with API Integration
# Plan on having this script take in an api key from the user and get the most up to date
# info directly from Van (Demographic info and Canvas Results)

# Set My List to Rayner's positive ID list
# Crosstab with [Precinct, Age, Likely Ethnicity, Rating For Voter: Responses] columns
# Filter
# Export to excel

# get the demographic data and label the columns
df = pd.DataFrame(pd.read_html('DemographicExport.xls')[0])
df.columns = ['Precinct','Age','LikelyEthn','idScore','TotalPeople']
df.drop(df.tail(1).index,inplace=True)

# get the total canvas numbers and label the columns
tf = pd.DataFrame(pd.read_html('CanvasResultsExport.xls')[0])

# get the total number of registered voters in each precinct 
# (according to van)
rf = pd.DataFrame(pd.read_html('TotalDistrictPopulationsExport.xls')[0])
rf.columns = ['Precinct','Age','LikelyEthn','TotalPeople']
rf.drop(df.tail(1).index,inplace=True)

# trim the scores to just the number
for i in range(0,len(df)):
    df.at[i,'idScore'] = int(df.at[i,'idScore'][:1])

In [2]:
# helper functions

# make sure the returned value is within the clamp boundaries
def clamp(num, min_value, max_value):
        num = max(min(num, max_value), min_value)
        return num
    
# return all the people in a given precinct
def in_precinct(prec, d):
    return d[d['Precinct'] == prec]

# return all the unique ages in a given set
def uniq_age(d):
    return list(d.Age.unique())

# return all the people in a given age range
def is_age(age, d):
    return d[d['Age'] == age]

# return all the unique ethnicities in a given precinct
def uniq_ethn(d):
    uniq = {}
    for le in d.LikelyEthn.unique():
        [ethn,conf] = le.split()
        uniq[ethn] = True
    return list(uniq.keys())

# return all the people of a given likely ethnicity
def is_ethn(ethn, d):
    return d[d.LikelyEthn.str.startswith(ethn)]

# return all the unique sexes in a given precinct
def uniq_sex(d):
    return []

# TODO: return all the people of a given sex
def is_sex(sex, d):
    return d

# return all the people of a given id score
def has_score(s, d):
    return d[d['idScore'] == s]

# return the total number of people in a given dataset
def how_many(d):
    return d.TotalPeople.sum()

# TODO: return the total number of doors knocked in a given precinct
def doors_knocked(p, d):
    return d[d['Precinct'] == p]['Canvassed']

# return our canvas success rate
def success_rate(p, d):
    return str(d[d['Precinct'] == p]['%.4']).split()[1]

# how many total positive ids
print('total pos ids: ' + str(how_many(df)))

# how many positive ids in district 04-012 are white voters age 65+
how_many(in_precinct('04-005',is_ethn('White', is_age('65+', df))))

total pos ids: 1283


7

In [3]:
# Visualization
import folium as fo

# Creates a new map centered on Baltimore County TODO: Center on District 10
m = fo.Map(location=[39.4107728, -76.7546931],zoom_start=11)

# Precinct to coordinate mapping for District 10
# these coordinates are based on the polling locations on this map. They are not 100% within the precincts
# they serve, but it is representative enough for now
# https://resources.baltimorecountymd.gov/Documents/infotech/gis/Elections/LegislativeDistrict10.pdf
ptc = {
    '01-002': [39.313476, -76.764579], # Chadwick Elementary
    '02-003': [39.322916, -76.738234], # Featherbed Lane Elementary
    '02-005': [39.345226, -76.752872], # Hebbville Elementary
    '02-006': [39.357004, -76.749675], # Milford Mill Academy
    '02-007': [39.365083, -76.756345], # Old Court Middle
    '02-009': [39.358458, -76.763998], # Scotts Branch Elementary
    '02-010': [39.353970, -76.779429], # Winfield Elementary
    '02-011': [39.341163, -76.760180], # comm support for the deaf (couldn't find, approximated)
    '02-012': [39.362057, -76.788868], # Liberty Senior Center
    '02-013': [39.368960, -76.779506], # Church Lane Elementary
    '02-014': [39.369960, -76.779506], # Church Lane Elementary + north
    '02-016': [39.380684, -76.786752], # Randallstown High + southeast
    '02-017': [39.381684, -76.785752], # Randallstown High
    '02-018': [39.375521, -76.730662], # Hernwood Elementary
    '02-019': [39.341322, -76.854536], # Granite Presbyterian Church
    '02-020': [39.389737, -76.770750], # Deer Park Elementary + West
    '02-021': [39.444167, -76.792881], # West Chapel Methodist Church
    '04-002': [39.444190, -76.792937], # Timber Grove Elementary
    '04-004': [39.467385, -76.808991], # Glyndon Elementary + East
    '04-005': [39.467385, -76.810991], # Glyndon Elementary
    '04-006': [39.455437, -76.815877], # Reisterstown Elementary
    '04-007': [39.464532, -76.832590], # Franklin Middle + West
    '04-008': [39.463669, -76.826834], # Chatsworth School
    #'04-009': [39.3, -76.7], # Boring Volunteer Fire Company
    '04-010': [39.463168, -76.832504], # Franklin Elementary
    '04-011': [39.536636, -76.728292], # Butler Volunteer Fire Company + BIG Southwest
    '04-012': [39.439423, -76.809278], # Cedarmere Elementary
    '04-013': [39.454102, -76.727272], # Chestnut Ridge Volunteer Fire Company
    '05-002': [39.600902, -76.776995]  # Fifth District Elementary
    
    # TODO Add 44b?
}

# Stores data for each precinct
ps = {}

# generate the gps coordinates for a given precinct
def gen_coords(p):
    ps[p]['coords'] = ptc[p]
    return ptc[p]

# generate the default text of a popup for a given precinct
def gen_popup_text(p):
    d = df[df['Precinct'] == p]
    trv = how_many(in_precinct(p, rf[rf['Precinct'] == p]))
    t = how_many(in_precinct(p,df))
    s = success_rate(p, tf)
    text = p+': '+str(t)+' positive ids with a success rate of '+s+'\n'
    text = text+'\nTotal registered voters: '+str(trv)+'\n'

    # add ethnicity readout
    text = text + '\nLikely Ethnicities\n\n'
    for ethn in uniq_ethn(d):
        te = how_many(is_ethn(ethn,d))
        prop = round((te * 100.0) / t, 2)
        text = text + '| ' + ethn + ' | ' + str(te) + ' | ' + str(prop) + '% |\n\n'
        
    # add age readout
    text = text + '\nAge\n\n'
    for age in uniq_age(d):
        ta = how_many(is_age(age,d))
        prop = round((ta * 100.0) / t, 2)
        text = text + '| ' + age + ' | ' + str(ta) + ' | ' + str(prop) + '% |\n\n'
    
    ps[p]['descText'] = text
    ps[p]['success'] = s
    return text

# generate the size for a given precinct
def gen_radius(p):
    # make the radius proportional to our success rate
    r = float(success_rate(p,tf)[:-1]) * how_many(in_precinct(p,df)) / 40 + 5
    ps[p]['radius'] = r
    return r

# Generate the color for a given precinct
def gen_color(p):
    red = "00"
    green = "00"
    blue = hex(clamp(how_many(in_precinct(p,df)) * 3, 0, 255))[2:]
    if len(blue) < 2:
        blue = "0" + blue
    rgb = "#" + red + green + blue
    ps[p]['color'] = rgb
    return rgb

# For every precinct we have the lat/long of, put its information on the map
for p in ptc:
    if p in df.Precinct.unique():
        ps[p] = {} # create space to store each precinct's data
        ps[p]['population'] = how_many(in_precinct(p,rf))
        ps[p]['total'] = how_many(in_precinct(p,df)) # total number of people in precinct
        fo.CircleMarker(
            tooltip=p, # text when hovered over
            location=ptc[p], # lat/long coordinates
            popup=gen_popup_text(p), # text when clicked
            radius=gen_radius(p), # size of marker
            color=gen_color(p), # color of marker as hex
            fill=True, # fill the marker?
            fill_color=gen_color(p) # what color to fill the marker with
        ).add_to(m)
    
m

In [4]:
# Additional Analysis

In [5]:
# total people canvassed
t = how_many(df)
out = '*Total Positive IDs: ' + str(t) + '*\n'

# how many total of each age group have we canvassed
out = out + '\n\n* Age Totals\n'
for age in uniq_age(df):
    # total people of a given age canvassed
    ta = how_many(is_age(age, df))
    # proportion of total canvassed population that is the given age
    prop = round((ta * 100.0) / t, 2)
    out = out+'| ' + age + ' | ' + str(ta) + ' | ' + str(prop) + '% |\n'
    
# how many total of each likely ethnicity have we canvassed
out=out+'\n\n* Likely Ethnicity Totals\n'
for ethn in uniq_ethn(df):
    # total people of a given age canvassed
    te = how_many(is_ethn(ethn, df))
    # proportion of total canvassed population that is the given age
    prop = round((te * 100.0) / t, 2)
    out = out+'| ' + ethn + ' | ' + str(te) + ' | ' + str(prop) + '% |\n'
    
# IDs by Precinct
out=out+'\n\n* Precincts by # Positive IDs\n'
# sort the precincts by number of positive ids and print the table
sorted_ps=dict(sorted(ps.items(),key= lambda x: x[1]['total'],reverse=True))
for p in sorted_ps:
    out=out+'** ' + ps[p]['descText']
    
# sort the precincts by success rate and print the table
out=out+'\n%\n%\n\n* Precincts by % Positive IDs\n'
sorted_ps=dict(sorted(ps.items(),key= lambda x: int(x[1]['success'][:-1]),reverse=True))
for p in sorted_ps:
    out=out+'** ' + ps[p]['descText']

# Readouts to my org file
org = open("writeup.org", "w")
org.write(out)
org.close()